# 06 · Supervised Modeling

Train 5 classifiers + 2 regressors with StratifiedGroupKFold (groups = `disaster_number`) and SMOTE in an imblearn pipeline. Tune RF with GridSearchCV, XGB with Optuna. Evaluate on the VAL split.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
import joblib
from sklearn.preprocessing import LabelEncoder
from src.modeling import (
    build_preprocessor, build_pipeline, get_classifiers, get_regressors,
    cv_score, tune_rf_grid, tune_xgb_optuna, regression_eval, classification_eval,
)
from config import CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES
df = pd.read_csv(PROC / 'abt_with_clusters.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
features = (FEATURE_GROUPS['demographics']+FEATURE_GROUPS['svi']
    +FEATURE_GROUPS['food_access']+FEATURE_GROUPS['flood']
    +FEATURE_GROUPS['storm']+['cluster_label','state'])
features = [f for f in features if f in df.columns]
X, y, g = df[features], df[TARGET_CLASS_COL], df['disaster_number']
le = LabelEncoder().fit(SEVERITY_LABELS); y_enc = le.transform(y)
tr = df['train_test_split']=='TRAIN'; va = df['train_test_split']=='VAL'

# Filter preprocessor feature lists to columns actually present in X
avail = set(X.columns)
cont = [c for c in CONTINUOUS_FEATURES if c in avail]
if 'cluster_label' in avail and 'cluster_label' not in cont:
    cont.append('cluster_label')
binf = [c for c in BINARY_FEATURES if c in avail]
catf = [c for c in CATEGORICAL_FEATURES if c in avail]
pre = build_preprocessor(continuous=cont, categorical=catf, binary=binf)
print('using', len(cont), 'cont +', len(catf), 'cat +', len(binf), 'bin features')

using 21 cont + 2 cat + 1 bin features


## Cross-val on TRAIN split

In [3]:
results = {}
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre)
    mean, arr = cv_score(pipe, X[tr], pd.Series(y_enc[tr]), g[tr])
    results[name] = mean
    print(f'{name:6s}  CV F1-weighted = {mean:.3f}  (folds: {arr.round(3)})')

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


rf      CV F1-weighted = 0.339  (folds: [0.479 0.267 0.271])


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


svm     CV F1-weighted = 0.105  (folds: [0.091 0.081 0.142])


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


...

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261

lr      CV F1-weighted = 0.333  (folds: [0.516 0.332 0.153])


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


xgb     CV F1-weighted = 0.287  (folds: [0.454 0.23  0.178])


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Tune RF + XGBoost

In [4]:
rf_gs = tune_rf_grid(X[tr], pd.Series(y_enc[tr]), g[tr], preprocessor=pre)
print('RF best:', rf_gs.best_params_, rf_gs.best_score_)

Fitting 3 folds for each of 27 candidates, totalling 81 fits
RF best: {'model__max_depth': 20, 'model__min_samples_leaf': 1, 'model__n_estimators': 100} 0.3616253996054659


In [5]:
try:
    study = tune_xgb_optuna(X[tr], pd.Series(y_enc[tr]), g[tr], n_trials=50, preprocessor=pre)
    print('XGB best:', study.best_params, study.best_value)
except Exception as e:
    print('XGB tuning skipped:', e); study = None

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-24 18:45:52,000] A new study created in memory with name: no-name-33f00245-8e19-41d9-87a7-865e6e56ce09
  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\L

[I 2026-04-24 18:46:18,586] Trial 0 finished with value: 0.37331746196293364 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746}. Best is trial 0 with value: 0.37331746196293364.


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 0. Best value: 0.373317:   4%|▍         | 2/50 [00:37<13:58, 17.46s/it]

[I 2026-04-24 18:46:29,661] Trial 1 finished with value: 0.3046805662175864 and parameters: {'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.19030368381735815, 'subsample': 0.8404460046972835, 'colsample_bytree': 0.8832290311184181}. Best is trial 0 with value: 0.37331746196293364.


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 2. Best value: 0.374193:   6%|▌         | 3/50 [00:48<11:27, 14.63s/it]

...

[I 2026-04-24 19:04:36,978] Trial 47 finished with value: 0.34932417747428507 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.09382099209637147, 'subsample': 0.7602672869114845, 'colsample_bytree': 0.8699815531365149}. Best is trial 24 with value: 0.41701888829424366.


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 24. Best value: 0.417019:  98%|█████████▊| 49/50 [19:09<00:24, 24.05s/it]

[I 2026-04-24 19:05:01,704] Trial 48 finished with value: 0.3152324179730261 and parameters: {'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.020241592544518253, 'subsample': 0.9489575695397194, 'colsample_bytree': 0.6111036996674017}. Best is trial 24 with value: 0.41701888829424366.


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 24. Best value: 0.417019: 100%|██████████| 50/50 [19:38<00:00, 23.58s/it]

[I 2026-04-24 19:05:30,756] Trial 49 finished with value: 0.3676637189851659 and parameters: {'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.11915287815220028, 'subsample': 0.8268197035847996, 'colsample_bytree': 0.910881970641672}. Best is trial 24 with value: 0.41701888829424366.
XGB best: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.21559005801510014, 'subsample': 0.7910037554626388, 'colsample_bytree': 0.7507983883065513} 0.41701888829424366


## Fit on TRAIN, evaluate on VAL — WITH cluster_label

In [6]:
val_scores = {}
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre).fit(X[tr], y_enc[tr])
    pred = pipe.predict(X[va])
    val_scores[name+'_withcluster'] = classification_eval(y_enc[va], pred)['f1_weighted']
    print(f'{name} WITH cluster  F1={val_scores[name+"_withcluster"]:.3f}')

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


rf WITH cluster  F1=0.159


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


svm WITH cluster  F1=0.050


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


nb WITH cluster  F1=0.583


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261

lr WITH cluster  F1=0.050
xgb WITH cluster  F1=0.085


c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Same evaluation WITHOUT cluster_label

In [7]:
features_noc = [f for f in features if f != 'cluster_label']
cont_noc = [c for c in cont if c != 'cluster_label' and c in features_noc]
pre_noc = build_preprocessor(continuous=cont_noc, categorical=catf, binary=binf)
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre_noc).fit(X[tr][features_noc], y_enc[tr])
    pred = pipe.predict(X[va][features_noc])
    val_scores[name+'_nocluster'] = classification_eval(y_enc[va], pred)['f1_weighted']
cmp = pd.DataFrame({
  'with_cluster': {k.replace('_withcluster',''): v for k,v in val_scores.items() if k.endswith('_withcluster')},
  'without_cluster': {k.replace('_nocluster',''): v for k,v in val_scores.items() if k.endswith('_nocluster')},
}); cmp['delta'] = cmp['with_cluster'] - cmp['without_cluster']; cmp

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, 

,with_cluster,without_cluster,delta
rf,0.158634,0.143663,0.014971
svm,0.050079,0.045680,0.004399
nb,0.583436,0.582638,0.000798
lr,0.049672,0.060753,-0.011081
xgb,0.085003,0.136574,-0.051571


## Regression variants

In [8]:
from sklearn.pipeline import Pipeline as SkPipeline
regs_scores = {}
for name, reg in get_regressors().items():
    pipe_r = SkPipeline([('pre', build_preprocessor(continuous=cont, categorical=catf, binary=binf)), ('reg', reg)])
    pipe_r.fit(X[tr], df.loc[tr, TARGET_COL])
    pred = pipe_r.predict(X[va])
    regs_scores[name] = regression_eval(df.loc[va, TARGET_COL], pred)
regs_scores

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'rf': {'rmse': 59.418821904409675,
  'mae': 31.262866197764517,
  'r2': 0.014969208257851196},
 'xgb': {'rmse': 62.02110198944255,
  'mae': 34.43644449764763,
  'r2': -0.07320006427083148}}

## Save best classifier + regressor

In [9]:
best_name = max({k:v for k,v in val_scores.items() if k.endswith('_withcluster')}.items(), key=lambda kv: kv[1])[0].replace('_withcluster','')
print('best classifier:', best_name)
best_clf = build_pipeline(get_classifiers()[best_name], preprocessor=pre).fit(X[tr], y_enc[tr])
joblib.dump({'pipe': best_clf, 'label_encoder': le, 'features': features},
            MODELS / 'best_classifier.pkl')
best_reg_name = min(regs_scores.items(), key=lambda kv: kv[1]['rmse'])[0]
best_reg = SkPipeline(
    [('pre', build_preprocessor(continuous=cont, categorical=catf, binary=binf)),
     ('reg', get_regressors()[best_reg_name])])
best_reg.fit(X[tr], df.loc[tr, TARGET_COL])
joblib.dump({'pipe': best_reg, 'features': features}, MODELS / 'best_regressor.pkl')
print('saved', MODELS / 'best_classifier.pkl', '/', MODELS / 'best_regressor.pkl')

best classifier: nb
saved C:\Users\chaitanya\Documents\ML Project\models\best_classifier.pkl / C:\Users\chaitanya\Documents\ML Project\models\best_regressor.pkl
